<a href="https://colab.research.google.com/github/Zattyla/K-UVR-Colab/blob/main/K-UVR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌊 K-UVR | Audio Separator (v1.0)
*Advanced Vocal & Instrumental Extraction Pipeline*

### 🚀 How to use:
1.  **Select GPU:** Ensure you are using a **T4 GPU** (`Runtime` -> `Change runtime type`).
2.  **Run Cell 1:** Install the core engine and dependencies.
3.  **Run Cell 2:** Download the professional model library (Kim, MDX-Net, etc.).
4.  **Upload Audio:** Click the **Folder icon** on the left and upload your song to the `inputs` folder.
5.  **Run Cell 3:** Enter your filename, choose a model, and toggle **DEREVERB** for cleaner AI covers.

### 🔬 Model Guide:
- **Kim_Vocal_2:** Your "Go-to" model. Incredible vocal isolation.
- **MDX_UVR_Vocal_FT:** Use this if the song has heavy backing vocals or complex harmonies.
- **UVR_Inst_HQ_3:** Use this to get a studio-quality instrumental track for karaoke or mixing.
- **De-reverb:** Automatically cleans the "hall effect" or echo from the vocals.

---

In [ ]:
# @title 1. Workspace Initialization
import os
from google.colab import drive

# Create folders for the pipeline
folders = ["inputs", "outputs", "models"]
for folder in folders:
    os.makedirs(f"/content/{folder}", exist_ok=True)

print("✅ Workspace folders created: /inputs, /outputs, /models")

In [ ]:
# @title 2. Install UVR Engine
import os
import sys

print("🧹 Cleaning environment...")
# Removemos tudo o que pode causar conflito
!pip uninstall -y numpy audio-separator tensorflow numba

print("\n⏳ Installing compatible core dependencies...")
# Forçamos o NumPy 1.26.4 que é o "ponto doce" para IA de áudio
!pip install numpy==1.26.4 --quiet

print("⏳ Installing Audio-Separator...")
# Instalamos a versão estável
!pip install -q audio-separator[gpu]

print("\n⚙️ Configuring System PATH...")
# Garante que o Linux do Colab encontre o comando
os.environ['PATH'] += ":/usr/local/bin"

print("\n🔍 Final Verification:")
try:
    import audio_separator
    print(f"✅ Library 'audio_separator' imported successfully.")
    # Testa se o comando via terminal funciona
    !python3 -m audio_separator.utils.cli --version
except Exception as e:
    print(f"❌ Setup failed: {e}")

print("\n⚠️ IMPORTANT: If you see a 'RESTART SESSION' button, CLICK IT and run Step 2 again once.")

In [ ]:
# @title 3. Workspace Final Check
import os

# Create the models directory if it doesn't exist
os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/inputs", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

print("✅ Workspace Ready.")
print("💡 Models will be downloaded automatically during Step 4.")

In [ ]:
# @title 4. Run Audio Separation
import os

# @markdown ### 📂 File Settings
# @markdown Enter the filename of the song in '/inputs':
FILE_NAME = "song.mp3" # @param {type:"string"}

# @markdown ### 📝 Model Selection
# @markdown - **Kim_Vocal_2**: Standard high-quality vocals.
# @markdown - **UVR-MDX-NET-Voc_FT**: Best for clean backing vocals.
# @markdown - **UVR-MDX-NET-Inst_HQ_3**: Best for instrumentals.
MODEL_CHOICE = "Kim_Vocal_2" # @param ["Kim_Vocal_2", "UVR-MDX-NET-Voc_FT", "UVR-MDX-NET-Inst_HQ_3"]

# @markdown ### 🧹 Post-Processing
DEREVERB = True # @param {type:"boolean"}

input_path = f"/content/inputs/{FILE_NAME}"

if os.path.exists(input_path):
    print(f"🚀 Initializing Engine for {FILE_NAME}...")
    print(f"📦 Selected Model: {MODEL_CHOICE}.onnx")

    # Execution using the direct python module
    !python3 -m audio_separator.utils.cli "{input_path}" \
        --model_filename "{MODEL_CHOICE}.onnx" \
        --output_dir "/content/outputs" \
        --output_format WAV \
        --normalization 0.9 \
        --model_file_dir "/content/models"

    # De-reverb step
    if DEREVERB:
        print("\n🧹 Running De-reverb for cleaner vocals...")
        # Finding the generated vocal file
        vocal_files = [f for f in os.listdir("/content/outputs") if "Vocals" in f and f.endswith(".wav")]
        if vocal_files:
            latest_vocal = os.path.join("/content/outputs", vocal_files[-1])
            !python3 -m audio_separator.utils.cli "{latest_vocal}" \
                --model_filename "UVR-DeReverb-HQ.onnx" \
                --output_dir "/content/outputs" \
                --output_format WAV \
                --model_file_dir "/content/models"
            print("✅ De-reverb process complete.")

    print("\n✨ ALL DONE! Find your files in the '/outputs' folder.")
else:
    print(f"❌ ERROR: File '{FILE_NAME}' not found in /content/inputs.")
    print("Please upload your file to the inputs folder on the left sidebar.")

In [ ]:
# @title 5. Cleanup & GPU Refresh
import torch, gc

# Clear GPU cache to prevent OOM errors
torch.cuda.empty_cache()
gc.collect()

print("✨ GPU Memory has been cleared.")
print("📁 You can now download your files from the 'outputs' folder on the left sidebar.")

# 🛠️ Troubleshooting Guide (UVR Edition)

If the separation process fails, check the solutions below:

### 1. ❌ "GPU not found" or NVIDIA errors
* **Cause:** Your Colab runtime is set to CPU or your GPU quota has ended.
* **Solution:** Go to `Runtime` -> `Change runtime type` and select **T4 GPU**. If it's already selected and still fails, you may need to wait 12-24h for a quota refresh from Google.

### 2. ❌ "FileNotFoundError: song.mp3 not found"
* **Cause:** The filename in Cell 4 does not match the file you uploaded.
* **Solution:** Check the `inputs` folder on the left sidebar. Ensure the filename has **no spaces** or special characters.
    * *Bad:* `My Song.mp3`
    * *Good:* `mysong.mp3`

### 3. ❌ "Out of Memory (OOM)"
* **Cause:** The GPU memory is full from a previous process.
* **Solution:** Run **Cell 5 (Cleanup)** to refresh the VRAM. If it persists, go to `Runtime` -> `Restart Session`.

### 4. ❌ "Model Download Failed"
* **Cause:** Connection issues with HuggingFace or Curl.
* **Solution:** Run **Cell 3** again. If it fails, manually check if you have enough disk space in the `/content` directory.

### 5. 🔊 Vocals still have background noise?
* **Tip:** Try a different model in **Cell 4**. `MDX_UVR_Vocal_FT` is usually better for songs with very loud instruments or backing vocals.